# Tutorial 10: Genetic Algorithms

**Prerequisites:** T9 (Why Local Methods Fail)  
**Up Next:** T11 — Particle Swarm Optimization

---

## Concept

A **Genetic Algorithm (GA)** is inspired by natural selection. We maintain a *population* of candidate solutions ("individuals"). Each generation:

1. **Selection** — fitter individuals are more likely to be chosen as parents.
2. **Crossover** — pairs of parents combine their genes to produce offspring.
3. **Mutation** — small random perturbations add diversity.

Over many generations, the population converges toward high-quality regions of the search space while maintaining enough diversity to escape local minima.

## Key Takeaway

> **GA explores broadly via population diversity — selection drives the population toward good regions, while crossover and mutation prevent premature convergence to a single local minimum.**

## Math Core

For a population of size $N$ with individuals $\mathbf{x}_i \in \mathbb{R}^n$:

**Tournament selection:** pick $k$ individuals at random; the one with lowest $f$ wins.

**Blend crossover (BLX-$\alpha$):** given parents $\mathbf{p}_1, \mathbf{p}_2$,
$$\text{child}_j = \mathbf{p}_{1,j} + \beta_j \,(\mathbf{p}_{2,j} - \mathbf{p}_{1,j}), \quad \beta_j \sim U(-\alpha, 1+\alpha)$$

**Gaussian mutation:**
$$\mathbf{x}'_j = \mathbf{x}_j + \sigma \, \mathcal{N}(0, 1)$$

with probability $p_m$ per gene.

## Code

In [ ]:
from dms.mechanisms.fourbar import FourBar
from dms.curves import CompareCurves
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

L0, L1 = 1.0, 1.0
PX, PY = 0.5, 0.3
TARGETS = np.array([
    [1.0, 1.5], [0.75, 1.5], [0.5, 1.5],
    [0.25, 1.5], [0.0, 1.5], [-0.25, 1.5]
])
THETAS = np.linspace(np.deg2rad(60), np.deg2rad(120), len(TARGETS))

In [ ]:
def fourbar_fk(theta1, l0, l1, l2, l3):
    ax, ay = l1 * np.cos(theta1), l1 * np.sin(theta1)
    dx, dy = ax - l0, ay
    d = np.sqrt(dx**2 + dy**2)
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    beta = np.arctan2(dy, dx)
    theta3 = beta + alpha
    bx = l0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    theta2 = np.arctan2(by - ay, bx - ax)
    return theta2, theta3


def coupler_point(theta1, l2, l3):
    sol = fourbar_fk(theta1, L0, L1, l2, l3)
    if sol is None:
        return None
    theta2, _ = sol
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    ux, uy = np.cos(theta2), np.sin(theta2)
    return np.array([ax + PX * ux - PY * uy,
                     ay + PX * uy + PY * ux])


def objective(x):
    l2, l3 = x
    path = []
    for theta1 in THETAS:
        pt = coupler_point(theta1, l2, l3)
        if pt is None:
            return 1e3
        path.append(pt)
    path = np.array(path)
    return CompareCurves(path[:, 0], path[:, 1],
                         TARGETS[:, 0], TARGETS[:, 1])

### Objective landscape (for visualization)

In [ ]:
l2_vals = np.linspace(0.3, 2.5, 40)
l3_vals = np.linspace(0.3, 2.5, 40)
L2g, L3g = np.meshgrid(l2_vals, l3_vals)
Z = np.vectorize(lambda a, b: objective([a, b]))(L2g, L3g)

### Implementing a simple GA from scratch

In [ ]:
def run_ga(objective, bounds, pop_size=40, n_gen=80,
           tournament_k=3, crossover_alpha=0.5,
           mutation_rate=0.3, mutation_sigma=0.15, seed=42):
    """Simple real-coded Genetic Algorithm."""
    rng = np.random.default_rng(seed)
    n_dim = len(bounds)
    lo = np.array([b[0] for b in bounds])
    hi = np.array([b[1] for b in bounds])

    # Initialize population uniformly
    pop = rng.uniform(lo, hi, size=(pop_size, n_dim))
    fitness = np.array([objective(ind) for ind in pop])

    best_history = []
    pop_history = [pop.copy()]

    for gen in range(n_gen):
        new_pop = []
        # Elitism: keep the best individual
        best_idx = np.argmin(fitness)
        new_pop.append(pop[best_idx].copy())

        while len(new_pop) < pop_size:
            # Tournament selection — pick two parents
            parents = []
            for _ in range(2):
                candidates = rng.integers(0, pop_size, size=tournament_k)
                winner = candidates[np.argmin(fitness[candidates])]
                parents.append(pop[winner])

            # Blend crossover (BLX-alpha)
            p1, p2 = parents
            beta = rng.uniform(-crossover_alpha, 1 + crossover_alpha, size=n_dim)
            child = p1 + beta * (p2 - p1)

            # Gaussian mutation
            for j in range(n_dim):
                if rng.random() < mutation_rate:
                    child[j] += rng.normal(0, mutation_sigma)

            # Clip to bounds
            child = np.clip(child, lo, hi)
            new_pop.append(child)

        pop = np.array(new_pop[:pop_size])
        fitness = np.array([objective(ind) for ind in pop])
        best_history.append(fitness.min())
        pop_history.append(pop.copy())

    best_idx = np.argmin(fitness)
    return pop[best_idx], fitness[best_idx], best_history, pop_history

In [ ]:
bounds = [(0.3, 2.5), (0.3, 2.5)]
best_x, best_f, history, pop_history = run_ga(objective, bounds)

print(f'GA best solution: l2={best_x[0]:.4f}, l3={best_x[1]:.4f}')
print(f'GA best objective: {best_f:.4f}')

### Convergence plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history, 'b-', lw=1.5)
ax.set_xlabel('Generation')
ax.set_ylabel('Best objective')
ax.set_title('GA convergence')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()

### Population evolving on the contour plot

We show snapshots of the population at generations 0, 20, 50, and the final generation.

In [ ]:
snap_gens = [0, 20, 50, len(pop_history) - 1]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, g in zip(axes, snap_gens):
    ax.contourf(L2g, L3g, np.log1p(Z), levels=30, cmap='viridis', alpha=0.8)
    pop_g = pop_history[g]
    ax.scatter(pop_g[:, 0], pop_g[:, 1], c='red', s=15, edgecolors='white', linewidths=0.5)
    ax.set_title(f'Gen {g}')
    ax.set_xlabel(r'$\ell_2$')
    ax.set_ylabel(r'$\ell_3$')
    ax.set_xlim(0.3, 2.5)
    ax.set_ylim(0.3, 2.5)

fig.suptitle('GA population snapshots', fontsize=14)
plt.tight_layout()

### Compare GA to BFGS from a random start

In [ ]:
from scipy.optimize import minimize

rng_bfgs = np.random.default_rng(42)
x0 = rng_bfgs.uniform(0.3, 2.5, size=2)
res_bfgs = minimize(objective, x0, method='L-BFGS-B',
                    bounds=[(0.3, 2.5), (0.3, 2.5)])

print(f'BFGS (single start at [{x0[0]:.2f}, {x0[1]:.2f}]):')
print(f'  Solution: l2={res_bfgs.x[0]:.4f}, l3={res_bfgs.x[1]:.4f}, f={res_bfgs.fun:.4f}')
print(f'\nGA (population of 40, 80 generations):')
print(f'  Solution: l2={best_x[0]:.4f}, l3={best_x[1]:.4f}, f={best_f:.4f}')
print(f'\nGA found a {"better" if best_f < res_bfgs.fun else "worse"} solution.')

### Visualizing the GA result

In [ ]:
path_ga = np.array([coupler_point(t, best_x[0], best_x[1]) for t in THETAS])

mechanism = FourBar(L0, L1, best_x[0], best_x[1])
fig, ax = plt.subplots(figsize=(6, 5))
mechanism.plot(np.deg2rad(90), ax=ax)
ax.plot(TARGETS[:, 0], TARGETS[:, 1], 'r*', ms=10, label='targets')
ax.plot(path_ga[:, 0], path_ga[:, 1], 'b.-', label='coupler path')
ax.set_aspect('equal')
ax.legend()
ax.set_title(f'GA result: $\\ell_2$={best_x[0]:.3f}, $\\ell_3$={best_x[1]:.3f}, f={best_f:.4f}')
plt.tight_layout()

---

## Exercises

1. **Tournament size:** Change `tournament_k` from 3 to 7. Does stronger selection pressure speed up convergence? Does it risk premature convergence?

2. **Mutation rate:** Try `mutation_rate=0.0` (no mutation). What happens to population diversity after 50 generations?

3. **Population size:** Run with `pop_size=10` vs `pop_size=100`. Plot both convergence curves on the same axes.

4. **Hybrid approach:** Take the GA's best solution and use it as the starting point for BFGS. Does the objective improve further? This is a common strategy called a *memetic algorithm*.